# 16 — Convertible Bonds

A convertible bond can be exchanged for shares of the issuer's stock.

- **Binomial tree** pricing (BSM)
- Parity, conversion value, and bond floor
- Conversion premium
- Stock-price sensitivity via AD
- Delta (equity sensitivity) of a convertible

In [ ]:
BACKEND = "cpu"

import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), "..") if not os.getcwd().endswith("ql-jax") else ".")
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath("__file__")), ".."))

from notebooks._common import setup_backend, load_quantlib, compare_table, timed_ms
jax = setup_backend(BACKEND)
import jax.numpy as jnp
import numpy as np
QL = load_quantlib()

In [ ]:
from ql_jax.instruments.convertible_bond import ConvertibleBond
from ql_jax.engines.lattice.bsm import bsm_convertible_tree

FACE = 100.0
COUPON = 0.04
CONV_RATIO = 1.0   # 1 share per 100 face
T = 5.0
S = 100.0           # current stock price
r, q, sigma = 0.05, 0.0, 0.30

---
## 1. Instrument Setup

In [ ]:
cb = ConvertibleBond(
    face_value=FACE, coupon_rate=COUPON,
    conversion_ratio=CONV_RATIO, maturity=T,
    coupon_frequency=2)

print(f"Conversion price  : {cb.conversion_price:.2f}")
print(f"Parity at S={S}  : {cb.parity(S):.2f}")
print(f"Conversion value  : {cb.conversion_value(S):.2f}")

---
## 2. Binomial Tree Pricing

In [ ]:
price = float(bsm_convertible_tree(
    S=S, K=FACE/CONV_RATIO, T=T, r=r, q=q, sigma=sigma,
    face_value=FACE, coupon_rate=COUPON,
    conversion_ratio=CONV_RATIO, n_steps=200))

# Straight bond floor (no conversion)
coupon = FACE * COUPON / 2
bond_floor = sum(coupon * np.exp(-r * (0.5 * i)) for i in range(1, 11)) + FACE * np.exp(-r * T)

conversion_value = S * CONV_RATIO
premium = price - max(bond_floor, conversion_value)

print(f"Convertible bond price : {price:.6f}")
print(f"Bond floor             : {bond_floor:.6f}")
print(f"Conversion value       : {conversion_value:.2f}")
print(f"Premium                : {premium:.4f}")

---
## 3. Stock Price Sensitivity

In [ ]:
import matplotlib.pyplot as plt

def cb_price_fn(stock):
    return bsm_convertible_tree(stock, FACE/CONV_RATIO, T, r, q, sigma,
                                 FACE, COUPON, CONV_RATIO, n_steps=100)

delta_fn = jax.grad(cb_price_fn)

spots = jnp.linspace(50, 200, 60)
prices = [float(cb_price_fn(s)) for s in spots]
deltas = [float(delta_fn(s)) for s in spots]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

ax1.plot(np.array(spots), prices, 'b-', linewidth=2, label='Convertible')
ax1.plot(np.array(spots), [bond_floor] * len(spots), 'g--', label='Bond floor')
ax1.plot(np.array(spots), np.array(spots) * CONV_RATIO, 'r--', label='Conversion value')
ax1.set_xlabel('Stock Price'); ax1.set_ylabel('Bond Price')
ax1.set_title('Convertible Bond Profile'); ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(np.array(spots), deltas, 'm-', linewidth=2)
ax2.set_xlabel('Stock Price'); ax2.set_ylabel('Delta')
ax2.set_title('Convertible Bond Delta (AD)'); ax2.grid(True, alpha=0.3)

fig.tight_layout()
plt.show()

---
## 4. Tree Convergence

In [ ]:
steps_list = [20, 50, 100, 200, 500]
conv_prices = [float(bsm_convertible_tree(S, FACE/CONV_RATIO, T, r, q, sigma,
                FACE, COUPON, CONV_RATIO, n_steps=n)) for n in steps_list]

plt.figure(figsize=(8, 5))
plt.plot(steps_list, conv_prices, 'bo-', markersize=8)
plt.xlabel('Binomial Steps')
plt.ylabel('Price')
plt.title('Convertible Bond: Tree Convergence')
plt.grid(True, alpha=0.3)
plt.show()

for n, p in zip(steps_list, conv_prices):
    print(f"  {n:4d} steps → {p:.6f}")

---
## 5. Exercises

1. **Call provision**: Add `call_price=110` and observe how the price changes.
2. **Volatility sensitivity**: Plot price vs σ from 10% to 60% — how does the equity-like option component grow?
3. **Rho**: Compute $\partial V / \partial r$ via AD.

---
*End of Notebook 16*